In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Model 1- Meta MMS (facebook/mms-1b-all)

In [ ]:
# STEP 2: Install dependencies
!pip install -q transformers soundfile librosa jiwer pandas torch



In [ ]:
# STEP 3: Imports
import os
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from transformers import Wav2Vec2BertProcessor, Wav2Vec2BertForCTC
from jiwer import wer, cer


In [ ]:
# STEP 4: Load model
# STEP 4: Load the Meta MMS-1B-All multilingual ASR model
from transformers import Wav2Vec2ForCTC, AutoProcessor
import torch

model_id = "facebook/mms-1b-all"
processor = AutoProcessor.from_pretrained(model_id)
model = Wav2Vec2ForCTC.from_pretrained(model_id)

# Switch to Tigrinya adapter
processor.tokenizer.set_target_lang("tir")
model.load_adapter("tir")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/sample.csv", encoding="utf-8", engine="python", sep=",")
# Strip column names in case there are spaces
df.columns = df.columns.str.strip()

print("Columns:", df.columns.tolist())
print(df.head())


Columns: ['audio_path', 'text']
                                 audio_path  \
0  /content/drive/MyDrive/sample/sente1.wav   
1                                       NaN   

                                                text  
0  መፃረይን መወገዲ ረሳሕ ፈስስን ከተማ መቐለ ንምህናፅ ዘኽእል እምነ ኩርና...  
1                                                NaN  


In [ ]:
# STEP 6: Helper to chunk long audio
def chunk_audio(speech, chunk_size=16000*5):  # 5-second chunks
    return [speech[i:i+chunk_size] for i in range(0, len(speech), chunk_size)]


In [ ]:
# STEP 7: Robust ASR transcription function with chunking
def asr_transcribe(audio_path):
    try:
        # Load audio
        speech, sr = sf.read(audio_path)

        # Stereo → mono
        if speech.ndim == 2:
            speech = np.mean(speech, axis=1)

        # Resample to 16 kHz
        if sr != 16000:
            speech = librosa.resample(speech, orig_sr=sr, target_sr=16000)

        # Convert to float32 and normalize
        speech = speech.astype(np.float32)
        speech = speech / max(np.max(np.abs(speech)), 1e-9)

        # Process in chunks
        full_transcription = ""
        for chunk in chunk_audio(speech):
            inputs = processor(chunk, sampling_rate=16000, return_tensors="pt", padding=True)
            input_values = inputs["input_values"].to(device)

            with torch.no_grad():
                logits = model(input_values).logits

            # Decode chunk
            predicted_ids = torch.argmax(logits, dim=-1)
            chunk_text = processor.batch_decode(predicted_ids)[0].strip()
            full_transcription += chunk_text + " "

        return full_transcription.strip()

    except Exception as e:
        print(f"Error transcribing {audio_path}: {e}")
        return ""


In [ ]:
# Strip spaces and convert everything to string
df["audio_path"] = df["audio_path"].astype(str).str.strip()

# Drop rows where audio_path is empty or 'nan'
df = df[df["audio_path"].str.lower() != 'nan'].reset_index(drop=True)

# Verify
print(df["audio_path"].tolist())


['/content/drive/MyDrive/sample/sente1.wav']


In [ ]:
predicted = []
for i, row in df.iterrows():
    path = row["audio_path"]
    if not os.path.exists(path):
        predicted.append("")
        continue
    print(f"[{i}] Transcribing:", path)
    # Load & preprocess audio
    speech, sr = sf.read(path)
    if speech.ndim == 2:
        speech = np.mean(speech, axis=1)
    if sr != 16000:
        speech = librosa.resample(speech, orig_sr=sr, target_sr=16000)
    speech = speech.astype(np.float32)
    speech = speech / max(np.max(np.abs(speech)), 1e-9)
    # Process with AutoProcessor
    inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)
    input_values = inputs["input_values"].to(device)
    with torch.no_grad():
        logits = model(input_values).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    text = processor.batch_decode(predicted_ids)[0].strip()
    predicted.append(text)
df["predicted"] = predicted
# Compute WER/CER
from jiwer import wer, cer
df["WER"] = df.apply(lambda r: wer(r["text"], r["predicted"]), axis=1)
df["CER"] = df.apply(lambda r: cer(r["text"], r["predicted"]), axis=1)
# Show results
for i, row in df.iterrows():
    print("="*80)
    print(f"Sample {i}")
    print("Ref :", row["text"])
    print("Pred:", row["predicted"])
    print("WER:", row["WER"], "CER:", row["CER"])

[0] Transcribing: /content/drive/MyDrive/sample/sente1.wav
Sample 0
Ref : መፃረይን መወገዲ ረሳሕ ፈስስን ከተማ መቐለ ንምህናፅ ዘኽእል እምነ ኩርናዕ ሎሚ ልለት ፕረዚንደት ግዝያዊ ምምሕዳር ታደሰ ወረደ ኣብ ዘለሉ ከምዝተነበረ ተሓቢሩ።
Pred: መፃረይን መወገድ ረሳሕ ፍሳስን ኸተማ መቐለን ምህናጽ ዘኽእልእምነዂርና ሎ ምዕለት ጵሬዝደንት ጊዜምምሕዳር ትግራይ ኣብዝተረኸበሉ ኸም ዝተነበረ ትሓቢሩ
WER: 0.9047619047619048 CER: 0.39603960396039606


In [ ]:
# STEP 11: Summary
print("Average WER:", df["WER"].mean())
print("Average CER:", df["CER"].mean())


Average WER: 0.9047619047619048
Average CER: 0.39603960396039606


In [ ]:
print(df.columns.tolist())


['audio_path', 'text', 'predicted', 'WER', 'CER']


In [ ]:
# Model 2: badrex/w2v-bert-2.0-tigrinya-asr

In [ ]:
from transformers import pipeline
import torch


In [ ]:
# Make sure audio paths are strings and clean
df["audio_path"] = df["audio_path"].astype(str).str.strip()
df = df[df["audio_path"].str.lower() != "nan"].reset_index(drop=True)

# List of audio files
audio_paths = df["audio_path"].tolist()


In [ ]:
from transformers import pipeline
import torch
from jiwer import wer, cer

print("\n=== Running Badrex ASR ===")
asr_badrex = pipeline(
    "automatic-speech-recognition",
    model="badrex/w2v-bert-2.0-tigrinya-asr",
    device=0 if torch.cuda.is_available() else -1
)

def asr_badrex_fn(audio_path):
    try:
        transcription = asr_badrex(audio_path)["text"]
        return transcription
    except Exception as e:
        print(f"Error transcribing {audio_path}: {e}")
        return ""

# Run on all audio paths
df["Badrex_predicted"] = [asr_badrex_fn(p) for p in audio_paths]

# Compute WER/CER
df["Badrex_WER"] = df.apply(lambda r: wer(r["text"], r["Badrex_predicted"]), axis=1)
df["Badrex_CER"] = df.apply(lambda r: cer(r["text"], r["Badrex_predicted"]), axis=1)

# Print actual vs predicted
for i, row in df.iterrows():
    print("="*80)
    print(f"[Sample {i}]")
    print("Actual   :", row["text"])
    print("Predicted:", row["Badrex_predicted"])
    print("WER:", row["Badrex_WER"], "CER:", row["Badrex_CER"])



=== Running Badrex ASR ===


Device set to use cuda:0


[Sample 0]
Actual   : መፃረይን መወገዲ ረሳሕ ፈስስን ከተማ መቐለ ንምህናፅ ዘኽእል እምነ ኩርናዕ ሎሚ ልለት ፕረዚንደት ግዝያዊ ምምሕዳር ታደሰ ወረደ ኣብ ዘለሉ ከምዝተነበረ ተሓቢሩ።
Predicted: መፃረይን ዎገዲ ረሳሕ ፈሳሲ ከተማ መቐለ ንምህናፅ ዘኽእል እምኒ ኩርናዕ ሎም ዕለት ፕረዚዳንት ግዝያዊ ምምሕዳር ትግራይ ኣብ ዝተረከበሉ ከም ዝተነበረ ተሓቢሩ
WER: 0.5714285714285714 CER: 0.2376237623762376


In [ ]:
# Model 3: Samuael/tigrinya-asr-characters

In [ ]:
print("\n=== Running Samuael ASR (character-level) ===")
asr_samuael = pipeline(
    "automatic-speech-recognition",
    model="Samuael/tigrinya-asr-characters",
    device=0 if torch.cuda.is_available() else -1
)

def asr_samuael_fn(audio_path):
    try:
        transcription = asr_samuael(audio_path)["text"]
        return transcription
    except Exception as e:
        print(f"Error transcribing {audio_path}: {e}")
        return ""

# Run on all audio paths
df["Samuael_predicted"] = [asr_samuael_fn(p) for p in audio_paths]

# Compute WER/CER
df["Samuael_WER"] = df.apply(lambda r: wer(r["text"], r["Samuael_predicted"]), axis=1)
df["Samuael_CER"] = df.apply(lambda r: cer(r["text"], r["Samuael_predicted"]), axis=1)

# Print actual vs predicted
for i, row in df.iterrows():
    print("="*80)
    print(f"[Sample {i}]")
    print("Actual   :", row["text"])
    print("Predicted:", row["Samuael_predicted"])
    print("WER:", row["Samuael_WER"], "CER:", row["Samuael_CER"])



=== Running Samuael ASR (character-level) ===


Device set to use cuda:0


[Sample 0]
Actual   : መፃረይን መወገዲ ረሳሕ ፈስስን ከተማ መቐለ ንምህናፅ ዘኽእል እምነ ኩርናዕ ሎሚ ልለት ፕረዚንደት ግዝያዊ ምምሕዳር ታደሰ ወረደ ኣብ ዘለሉ ከምዝተነበረ ተሓቢሩ።
Predicted: መጻረይን መወገድ ረሳሕፈሳስን  ከተማዋ መቀለን ምሕና ጽእዝ ኽእልእምነኹርና እሎሚያ እለት ፍሬዝዳንቲ ጊዜዋ መምሕዳር ትግራይ ኣብ ሕዝ ተረኸበሉ ከምዚ እተነበረት ሓቢሩ
WER: 1.0 CER: 0.5148514851485149


In [ ]:
# Model 4: Meta Omnilingual ASR

In [ ]:

# Install stable PyTorch + Omnilingual ASR + dependencies
!pip install --upgrade pip
!pip install torch==2.9.0+cu121 torchvision==0.15.2+cu121 torchaudio==2.9.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121
!pip install omnilingual-asr librosa soundfile pandas jiwer

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement torch==2.9.0+cu121 (from versions: 2.2.0, 2.2.0+cu121, 2.2.1, 2.2.1+cu121, 2.2.2, 2.2.2+cu121, 2.3.0, 2.3.0+cu121, 2.3.1, 2.3.1+cu121, 2.4.0, 2.4.0+cu121, 2.4.1, 2.4.1+cu121, 2.5.0, 2.5.0+cu121, 2.5.1, 2.5.1+cu121, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1)
ERROR: No matching distribution found for torch==2.9.0+cu121


In [ ]:
import os
import torch
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from jiwer import wer, cer


In [ ]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Initialize pipeline correctly
asr = ASRInferencePipeline(
    model_card="omniASR_CTC_1B",  # CTC variant of the Meta Omnilingual model
    device=device
)


parameter load: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━   785/807  97% 0:00:01

In [ ]:
csv_path = "/content/drive/MyDrive/sample.csv"
df = pd.read_csv(csv_path)

# Drop empty rows
df = df.dropna(subset=["audio_path", "text"])
df.reset_index(drop=True, inplace=True)

print(df.head())


                                 audio_path  \
0  /content/drive/MyDrive/sample/sente1.wav   

                                                text  
0  መፃረይን መወገዲ ረሳሕ ፈስስን ከተማ መቐለ ንምህናፅ ዘኽእል እምነ ኩርና...  


In [ ]:
# Step 1: Install dependencies (if not installed already)
!pip install -q omnilingual-asr jiwer soundfile librosa torch pandas

In [ ]:
# Step 2: Imports
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
import torch
import pandas as pd
from jiwer import wer, cer
import os

In [ ]:
# Step 3: Prepare CSV
# Ensure 'audio_path' column exists and is clean
df["audio_path"] = df["audio_path"].astype(str).str.strip()
df = df[df["audio_path"].str.lower() != "nan"].reset_index(drop=True)
audio_paths = df["audio_path"].tolist()


In [ ]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Initialize pipeline
asr = ASRInferencePipeline(
    model_card="omniASR_CTC_1B",
    device=device
)

audio_files = ["/content/drive/MyDrive/sample/sente1.wav"]

# Transcribe
results = asr.transcribe(audio_files, lang=["tir_Ethi"])

# Print actual vs predicted
for i, res in enumerate(results):
    print(f"[{i}] Transcribed text:", res)  # <--- just print res as string


parameter load: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━   761/807  94% 0:00:01

[0] Transcribed text: መጻረይን መወገዲ ረሳሕ ፈሳስን ከተማ መቐለ ን ምህናጽ እዘክእል እምነዂርና ሎም ዕለት ፕሬዚደንት ጊዜያዋ መምሕዳር ትግራይ ኣብ ዝተረኸበሉ ከም ዝተነበረ ተሓቢሩ


In [ ]:
import pandas as pd
from jiwer import wer, cer

df = pd.read_csv("/content/drive/MyDrive/sample.csv", encoding="utf-8")
df = df.dropna(subset=["audio_path", "text"])  # remove empty rows

predicted = asr.transcribe(df["audio_path"].tolist(), lang=["tir_Ethi"])
df["predicted"] = predicted
df["WER"] = df.apply(lambda r: wer(r["text"], r["predicted"]), axis=1)
df["CER"] = df.apply(lambda r: cer(r["text"], r["predicted"]), axis=1)

# Show results
for i, row in df.iterrows():
    print("="*80)
    print(f"[{i}] ACTUAL   :", row["text"])
    print(f"[{i}] PREDICTED:", row["predicted"])
    print(f"WER: {row['WER']}  CER: {row['CER']}")

print("Average WER:", df["WER"].mean())
print("Average CER:", df["CER"].mean())


[0] ACTUAL   : መፃረይን መወገዲ ረሳሕ ፈስስን ከተማ መቐለ ንምህናፅ ዘኽእል እምነ ኩርናዕ ሎሚ ልለት ፕረዚንደት ግዝያዊ ምምሕዳር ታደሰ ወረደ ኣብ ዘለሉ ከምዝተነበረ ተሓቢሩ።
[0] PREDICTED: መጻረይን መወገዲ ረሳሕ ፈሳስን ከተማ መቐለ ን ምህናጽ እዘክእል እምነዂርና ሎም ዕለት ፕሬዚደንት ጊዜያዋ መምሕዳር ትግራይ ኣብ ዝተረኸበሉ ከም ዝተነበረ ተሓቢሩ
WER: 0.8095238095238095  CER: 0.31683168316831684
Average WER: 0.8095238095238095
Average CER: 0.31683168316831684
